# Merged Ensemble Gold Evaluation

Ensemble of:
- Granite LoRA model (`granite_training_only` style)
- Clear-Non-Reply RoBERTa votes (`clear-non-reply-evaluation-roberta` style)

Special rule:
1. If **Direct Non-Reply** is voted by RoBERTa majority or Granite votes, final = `Direct Non-Reply`
2. Otherwise, use Granite vote majority (additive voting)


In [ ]:
!pip -q install -U transformers datasets peft bitsandbytes pandas scikit-learn

In [ ]:
import os
import json
import csv
from pathlib import Path
from collections import Counter

import torch
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

LABEL_MAP = {
    'Clear Reply': 'Direct Reply',
    'Clear Non-Reply': 'Direct Non-Reply',
    'Ambivalent': 'Indirect',
    'Ambivalent Reply': 'Indirect',
}

ALLOWED = ['Direct Reply', 'Direct Non-Reply', 'Indirect']


def map_label(raw):
    v = str(raw).strip()
    return LABEL_MAP.get(v, v if v in ALLOWED else 'Indirect')


def first_existing(paths):
    for p in paths:
        if Path(p).exists():
            return Path(p)
    return None


# Gold eval inputs (attach in Kaggle Input)
LABEL_CANDIDATES = [
    '/kaggle/input/datasets/gigibot/claritysemevalevaldataset/task1_eval_labels.txt',
    '/kaggle/input/claritysemevalevaldataset/task1_eval_labels.txt',
    '/kaggle/input/datasets/gigibot/evalsetsemevalpolitical/task1_eval_labels.txt',
    '/kaggle/input/evalsetsemevalpolitical/task1_eval_labels.txt',
    '/Users/andrearachetta/Downloads/task1_eval_labels.txt',
]
EVAL_CSV_CANDIDATES = [
    '/kaggle/input/datasets/gigibot/claritysemevalevaldataset/clarity_task_evaluation_dataset.csv',
    '/kaggle/input/claritysemevalevaldataset/clarity_task_evaluation_dataset.csv',
    '/kaggle/input/datasets/gigibot/evalsetsemevalpolitical/clarity_task_evaluation_dataset.csv',
    '/kaggle/input/evalsetsemevalpolitical/clarity_task_evaluation_dataset.csv',
    '/kaggle/working/CLARITY-SemEval-2026/dataset/clarity_task_evaluation_dataset.csv',
    'dataset/clarity_task_evaluation_dataset.csv',
]

# Granite adapter/checkpoint candidates (includes checkpoint-64 fallback)
ADAPTER_CANDIDATES = [
    '/kaggle/working/granite_lora_trained/checkpoint-64',
    '/kaggle/working/granite_lora_trained',
    '/kaggle/input/granite-lora-checkpoint/checkpoint-64',
    '/kaggle/input/granite-lora-checkpoint',
    '/Users/andrearachetta/Desktop/CLARITY-SemEval-2026/checkpoint64',
    '/Users/andrearachetta/Desktop/CLARITY-SemEval-2026/granite_clarity_finetuned/checkpoint-60',
    '/Users/andrearachetta/Desktop/CLARITY-SemEval-2026/granite_clarity_finetuned',
]

# RoBERTa vote source candidates (from your clear-non-reply notebook output)
ROBERTA_VOTES_CSV_CANDIDATES = [
    '/kaggle/input/clear-non-reply-preds/clear-non-reply-predictions-roberta.csv',
    '/kaggle/working/clear-non-reply-predictions-roberta.csv',
    '/Users/andrearachetta/Desktop/CLARITY-SemEval-2026/clear-non-reply-predictions-roberta.csv',
]

BASE_MODEL = 'ibm-granite/granite-3.2-8b-instruct'
LOAD_4BIT = True
GRANITE_NUM_VOTES = 3
GRANITE_TEMPERATURE = 0.6
MAX_NEW_TOKENS = 320
BATCH_SIZE = 2
OUT_DIR = Path('/kaggle/working/merged_ensemble_eval')
OUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
labels_path = first_existing(LABEL_CANDIDATES)
eval_csv_path = first_existing(EVAL_CSV_CANDIDATES)
adapter_path = first_existing(ADAPTER_CANDIDATES)
roberta_csv_path = first_existing(ROBERTA_VOTES_CSV_CANDIDATES)

if labels_path is None or eval_csv_path is None:
    raise FileNotFoundError('Missing gold eval files. Attach eval dataset as Kaggle Input.')
if adapter_path is None:
    raise FileNotFoundError('Missing Granite adapter/checkpoint. Attach or train first.')
if roberta_csv_path is None:
    raise FileNotFoundError('Missing RoBERTa votes CSV. Export from clear-non-reply notebook first.')

print('Gold labels:', labels_path)
print('Eval CSV:', eval_csv_path)
print('Granite adapter:', adapter_path)
print('RoBERTa votes CSV:', roberta_csv_path)

with labels_path.open('r', encoding='utf-8') as f:
    gold_labels = [map_label(line.strip()) for line in f if line.strip()]

with eval_csv_path.open('r', encoding='utf-8', newline='') as f:
    eval_rows = list(csv.DictReader(f))

examples = []
for r in eval_rows:
    idx_raw = str(r.get('index', '')).strip()
    idx = int(idx_raw) if idx_raw.isdigit() else len(examples)
    q = str(r.get('question') or r.get('interview_question') or '').strip()
    a = str(r.get('interview_answer') or r.get('answer') or '').strip()
    examples.append({'index': idx, 'question': q, 'answer': a})
examples = sorted(examples, key=lambda x: x['index'])

if len(examples) != len(gold_labels):
    raise ValueError(f'Mismatch: eval_rows={len(examples)} gold_labels={len(gold_labels)}')

roberta_df = pd.read_csv(roberta_csv_path)
if 'index' not in roberta_df.columns:
    raise ValueError('RoBERTa CSV must include index column')
for c in ['try_1', 'try_2', 'try_3']:
    if c not in roberta_df.columns:
        raise ValueError(f'Missing RoBERTa vote column: {c}')
roberta_df = roberta_df.sort_values('index').reset_index(drop=True)
if len(roberta_df) != len(examples):
    raise ValueError(f'RoBERTa rows ({len(roberta_df)}) != eval rows ({len(examples)})')

print('Alignment OK:', len(examples), 'samples')

In [ ]:
has_cuda = torch.cuda.is_available()
use_4bit = bool(LOAD_4BIT and has_cuda)

if not has_cuda and LOAD_4BIT:
    print('⚠️ CUDA not available: disabling 4-bit quantization and running in float32 on CPU/MPS.')

quant_config = None
if use_4bit:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_kwargs = {
    'torch_dtype': torch.bfloat16 if has_cuda else torch.float32,
}
if quant_config is not None:
    model_kwargs['quantization_config'] = quant_config
if has_cuda:
    model_kwargs['device_map'] = 'auto'

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **model_kwargs)
model = PeftModel.from_pretrained(model, str(adapter_path))

if not has_cuda:
    target_device = torch.device('mps' if (hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()) else 'cpu')
    model = model.to(target_device)

model.eval()

print('Granite loaded from adapter:', adapter_path)
print('Runtime:', 'CUDA' if has_cuda else ('MPS' if (hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()) else 'CPU'))
print('Using 4-bit:', use_4bit)


def build_prompt(q, a):
    return f"""You are analyzing political interview answers for clarity classification.

Question: {q}
Answer: {a}

Analyze the answer step-by-step:
1. Does it directly address the question?
2. Is it evasive or indirect?
3. Does it decline to answer?

Respond in JSON format:
{{
  \"reasoning\": \"...\",
  \"label\": \"Direct Reply|Direct Non-Reply|Indirect\"
}}"""


def parse_label(text):
    try:
        obj = json.loads(text)
        lbl = str(obj.get('label', 'Indirect')).strip()
        if lbl in ALLOWED:
            return lbl
    except Exception:
        pass
    low = text.lower()
    for lbl in ALLOWED:
        if lbl.lower() in low:
            return lbl
    return 'Indirect'


def granite_votes(question, answer, n_votes=3):
    prompt = build_prompt(question, answer)
    messages = [{'role': 'user', 'content': prompt}]
    txt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(txt, return_tensors='pt').to(model.device)

    votes = []
    for _ in range(max(1, n_votes)):
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=GRANITE_TEMPERATURE > 0,
                temperature=max(GRANITE_TEMPERATURE, 1e-5) if GRANITE_TEMPERATURE > 0 else 1.0,
                pad_token_id=tokenizer.eos_token_id,
            )
        resp = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        votes.append(parse_label(resp))
    return votes

In [ ]:
rows = []

for i, ex in enumerate(examples):
    q, a = ex['question'], ex['answer']
    idx = ex['index']
    true = gold_labels[idx]

    # Granite votes (3-class)
    gv = granite_votes(q, a, n_votes=GRANITE_NUM_VOTES)
    g_counter = Counter(gv)
    g_major = g_counter.most_common(1)[0][0]

    # RoBERTa votes (binary clear-non-reply detector)
    rr = roberta_df.iloc[idx]
    rv_bin = [int(rr['try_1']), int(rr['try_2']), int(rr['try_3'])]
    r_nonreply = 1 if sum(rv_bin) >= 2 else 0

    # Special rule requested:
    # - if Direct Non-Reply is voted -> take max/override
    # - else add (use Granite additive majority)
    if r_nonreply == 1 or ('Direct Non-Reply' in gv):
        pred = 'Direct Non-Reply'
    else:
        pred = g_major

    rows.append({
        'index': idx,
        'true': true,
        'pred': pred,
        'correct': int(pred == true),
        'granite_votes': gv,
        'granite_majority': g_major,
        'roberta_try_1': rv_bin[0],
        'roberta_try_2': rv_bin[1],
        'roberta_try_3': rv_bin[2],
        'roberta_nonreply_majority': r_nonreply,
        'question': q,
        'answer': a,
    })

    if (i + 1) % 10 == 0 or i == 0:
        running = sum(r['correct'] for r in rows) / len(rows)
        print(f'[{i+1}/{len(examples)}] running_acc={running:.4f}')

pred_df = pd.DataFrame(rows).sort_values('index').reset_index(drop=True)
acc = accuracy_score(pred_df['true'], pred_df['pred'])
macro_f1 = f1_score(pred_df['true'], pred_df['pred'], average='macro', zero_division=0)

print('\n=== Final Ensemble Metrics ===')
print(f'Accuracy: {acc:.4f}')
print(f'Macro-F1: {macro_f1:.4f}')
print('\nConfusion matrix:')
print(pd.crosstab(pred_df['true'], pred_df['pred'], rownames=['true'], colnames=['pred'], dropna=False))
print('\nClassification report:')
print(classification_report(pred_df['true'], pred_df['pred'], labels=ALLOWED, zero_division=0))

pred_df.to_csv(OUT_DIR / 'merged_ensemble_predictions.csv', index=False)
with (OUT_DIR / 'merged_ensemble_metrics.json').open('w', encoding='utf-8') as f:
    json.dump({
        'accuracy': float(acc),
        'macro_f1': float(macro_f1),
        'confusion_matrix': confusion_matrix(pred_df['true'], pred_df['pred'], labels=ALLOWED).tolist(),
        'labels': ALLOWED,
    }, f, indent=2)

print(f'\nSaved: {OUT_DIR / "merged_ensemble_predictions.csv"}')
print(f'Saved: {OUT_DIR / "merged_ensemble_metrics.json"}')